# Data Exploration
This notebook visualizes the custom hand digit dataset collected for this project. It covers image counts, sample images, bounding box annotations, and class distribution.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random
from pathlib import Path

imgs_dir = Path('../data/custom/images')
labels_dir = Path('../data/custom/labels')
crops_dir = Path('../data/custom/crops')

## Dataset Size

In [ ]:
images = list(imgs_dir.glob('*.jpg'))
labels = list(labels_dir.glob('*.txt'))
negatives = [l for l in labels if l.stat().st_size == 0]

print(f'Total images:      {len(images)}')
print(f'Total annotations: {len(labels)}')
print(f'Negative examples: {len(negatives)} (no hands)')
print()
for i in range(6):
    count = len(list((crops_dir / str(i)).glob('*.jpg')))
    print(f'Digit {i}: {count} crops')

## Sample Images

In [ ]:
samples = random.sample(images, 8)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, img_path in enumerate(samples):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    axes[i].imshow(img)
    axes[i].set_title(img_path.stem, fontsize=8)
    axes[i].axis('off')

plt.suptitle('Sample Images from Custom Dataset')
plt.tight_layout()
plt.show()

## Bounding Box Annotations
Verify that bounding box annotations are correctly aligned with the hands in the images.

In [ ]:
# Only show images that have at least one annotation
annotated = [l for l in labels if l.stat().st_size > 0]
samples = random.sample(annotated, 6)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, label_path in enumerate(samples):
    img_path = imgs_dir / (label_path.stem + '.jpg')
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]

    with open(label_path, 'r') as f:
        for line in f.readlines():
            values = line.strip().split()
            digit_class = int(values[0])
            x, y, w, h = float(values[1]), float(values[2]), float(values[3]), float(values[4])
            x1 = int((x - w/2) * W)
            y1 = int((y - h/2) * H)
            x2 = int((x + w/2) * W)
            y2 = int((y + h/2) * H)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 3)
            cv2.putText(img, str(digit_class), (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)

    axes[i].imshow(img)
    axes[i].set_title(label_path.stem, fontsize=8)
    axes[i].axis('off')

plt.suptitle('Sample Annotations — Bounding Boxes with Digit Class Labels')
plt.tight_layout()
plt.show()

## Class Distribution
Check that crop images are roughly balanced across digit classes.

In [ ]:
class_labels = ['Zero', 'One', 'Two', 'Three', 'Four', 'Five']
class_counts = [len(list((crops_dir / str(i)).glob('*.jpg'))) for i in range(6)]

plt.figure(figsize=(8, 5))
plt.bar(class_labels, class_counts, color='steelblue')
plt.title('Crop Distribution by Digit Class')
plt.xlabel('Number of Fingers')
plt.ylabel('Number of Crop Images')
for i, count in enumerate(class_counts):
    plt.text(i, count + 10, str(count), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## Sample Crops per Class
Visually inspect a few crops from each digit class.

In [ ]:
num_per_class = 4
fig, axes = plt.subplots(6, num_per_class, figsize=(12, 18))

for i in range(6):
    digit_crops = list((crops_dir / str(i)).glob('*.jpg'))
    samples = random.sample(digit_crops, min(num_per_class, len(digit_crops)))
    for j, crop_path in enumerate(samples):
        img = cv2.imread(str(crop_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        axes[i, j].imshow(img)
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_ylabel(class_labels[i], fontsize=12, rotation=0, labelpad=50)

plt.suptitle('Sample Crops per Digit Class', fontsize=14)
plt.tight_layout()
plt.show()